# Etapa 4 - Avaliação e Casos de Teste

Este notebook consome os artefatos gerados pelo Notebook 3 (`bandit_results.csv`, `bandit_metrics.json`, `bandit_model.pkl`, `context_rates.csv`) e é independente dele — não é preciso reexecutar o Notebook 3 na mesma sessão, só ter os arquivos gerados por ele em `data/processed/bank-term-deposit-subscription_eda/`.

Conteúdo:
1. **Métricas de avaliação** do modelo adaptativo (lift, intervalo de confiança, regret acumulado).
2. **Golden set de 5 clientes**, mostrando qual canal o modelo recomendaria para cada um e se a decisão faz sentido.

In [1]:
# MLflow setup
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
import mlflow
import sys
sys.path.insert(0, '..')
from app.mlflow_utils import log_notebook_execution, initialize_mlflow

initialize_mlflow()
mlflow.set_tracking_uri('http://localhost:5002')
mlflow.set_experiment('model-production')
print('MLflow initialized ✅\n')



MLflow initialized ✅



In [2]:
PROCESSED_DIR = Path('../data/processed/bank-term-deposit-subscription_eda')
RESULTS_PATH = PROCESSED_DIR / 'bandit_results.csv'
METRICS_PATH = PROCESSED_DIR / 'bandit_metrics.json'
BANDIT_MODEL_PATH = PROCESSED_DIR / 'bandit_model.pkl'
CONTEXT_RATES_PATH = PROCESSED_DIR / 'context_rates.csv'
GOLDEN_SET_PATH = PROCESSED_DIR / 'golden_set_results.csv'

required_files = [RESULTS_PATH, METRICS_PATH, BANDIT_MODEL_PATH, CONTEXT_RATES_PATH]
missing = [str(p) for p in required_files if not p.exists()]
if missing:
    raise FileNotFoundError(
        'Execute primeiro o Notebook 3 (ele salva os artefatos que faltam): ' + ', '.join(missing)
    )

In [3]:
results = pd.read_csv(RESULTS_PATH)
metrics = json.loads(METRICS_PATH.read_text(encoding='utf-8'))
context_rates = pd.read_csv(CONTEXT_RATES_PATH)

with BANDIT_MODEL_PATH.open('rb') as f:
    bandit = pickle.load(f)

arms = list(metrics['training_conversion_by_arm'])
real_rates = metrics['training_conversion_by_arm']

print(f"Baseline (regra fixa): {metrics['baseline_policy']} -> {metrics['baseline_arm']}")
print(f"Conversão baseline: {metrics['baseline_conversion']:.2%}")
print(f"Conversão Thompson Sampling: {metrics['thompson_conversion']:.2%}")
print(f'Bandit treinado carregado de: {BANDIT_MODEL_PATH}')

Baseline (regra fixa): regra fixa (sempre telephone) -> telephone
Conversão baseline: 10.44%
Conversão Thompson Sampling: 13.67%
Bandit treinado carregado de: ../data/processed/bank-term-deposit-subscription_eda/bandit_model.pkl


In [4]:
def contextual_rate(row, arm):
    match = context_rates[
        (context_rates['age_segment'] == row['age_segment'])
        & (context_rates['poutcome'] == row['poutcome'])
        & (context_rates['previous'] == row['previous'])
        & (context_rates['contact'] == arm)
    ]
    if match.empty:
        return real_rates[arm]

    local_mean = float(match.iloc[0]['mean'])
    local_count = float(match.iloc[0]['count'])
    prior_weight = 20.0
    return (local_mean * local_count + real_rates[arm] * prior_weight) / (local_count + prior_weight)

## Métricas de avaliação do modelo

Além da conversão observada na simulação (Etapa 3), consolidamos aqui as métricas relevantes para um bandit:

- **Conversão média** de cada política (já calculada).
- **Lift absoluto e relativo** do Thompson Sampling sobre o baseline.
- **Intervalo de confiança (95%)** da diferença de conversão, via teste de proporções — para saber se o ganho é estatisticamente confiável, não só um número pontual.
- **Regret acumulado** ao longo da simulação — mede quanto o bandit "deixou na mesa" em relação ao baseline a cada rodada.
- **Distribuição de escolhas por braço** — para verificar se o bandit não ficou preso a um único braço por acidente.

In [5]:
n_total = len(results)
n_thompson_success = int(results['thompson_reward'].sum())
n_baseline_success = int(results['baseline_reward'].sum())

p_thompson = n_thompson_success / n_total
p_baseline = n_baseline_success / n_total
p_pool = (n_thompson_success + n_baseline_success) / (2 * n_total)

se_pooled = np.sqrt(p_pool * (1 - p_pool) * (2 / n_total))
z_stat = (p_thompson - p_baseline) / se_pooled
p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))

se_diff = np.sqrt(p_thompson * (1 - p_thompson) / n_total + p_baseline * (1 - p_baseline) / n_total)
diff = p_thompson - p_baseline
ci_low, ci_high = diff - 1.96 * se_diff, diff + 1.96 * se_diff

cumulative_regret = (results['baseline_reward'] - results['thompson_reward']).cumsum()

evaluation_metrics = {
    'rodadas_avaliadas': n_total,
    'conversao_baseline': round(p_baseline, 4),
    'conversao_thompson': round(p_thompson, 4),
    'lift_absoluto_pp': round(diff * 100, 2),
    'lift_relativo_pct': round((p_thompson / p_baseline - 1) * 100, 1),
    'ic_95_diferenca_pp': [round(ci_low * 100, 2), round(ci_high * 100, 2)],
    'z_stat': round(float(z_stat), 2),
    'p_value': float(p_value),
    'regret_acumulado_final': int(cumulative_regret.iloc[-1]),
    'distribuicao_bracos_thompson': results['thompson_arm'].value_counts(normalize=True).round(4).to_dict(),
}

(PROCESSED_DIR / 'evaluation_metrics.json').write_text(json.dumps(evaluation_metrics, indent=2), encoding='utf-8')

print(f"Conversão baseline: {evaluation_metrics['conversao_baseline']:.2%}")
print(f"Conversão Thompson Sampling: {evaluation_metrics['conversao_thompson']:.2%}")
print(f"Lift: {evaluation_metrics['lift_absoluto_pp']:+.2f} p.p. ({evaluation_metrics['lift_relativo_pct']:+.1f}% relativo)")
print(f"IC 95% da diferença: {evaluation_metrics['ic_95_diferenca_pp']} p.p.")
print(f"z={evaluation_metrics['z_stat']}, p-valor={evaluation_metrics['p_value']:.2e}")
print(f"Regret acumulado ao final da simulação: {evaluation_metrics['regret_acumulado_final']}")
print(f"Distribuição de escolhas do Thompson: {evaluation_metrics['distribuicao_bracos_thompson']}")

Conversão baseline: 10.44%
Conversão Thompson Sampling: 13.67%
Lift: +3.23 p.p. (+31.0% relativo)
IC 95% da diferença: [np.float64(2.31), np.float64(4.15)] p.p.
z=6.9, p-valor=5.38e-12
Regret acumulado ao final da simulação: -312
Distribuição de escolhas do Thompson: {'cellular': 0.9899, 'telephone': 0.0101}


**Leitura:** o intervalo de confiança de 95% da diferença não cruza zero (todo ele é positivo), então o ganho do Thompson Sampling sobre a regra fixa é estatisticamente confiável, não coincidência da simulação. O regret acumulado negativo confirma que o bandit ficou sistematicamente à frente do baseline ao longo de toda a simulação, não só no resultado final.

## Golden set: 5 casos de teste

Um ponto importante para interpretar corretamente os resultados abaixo: o Thompson Sampling implementado aqui é um **bandit global**, não contextual — ele decide com base na crença agregada sobre `cellular` vs `telephone`, e essa crença é a mesma independentemente do perfil do cliente. Ou seja, **é esperado que a oferta recomendada seja a mesma (ou quase) para os 5 clientes**, porque hoje o modelo não personaliza por segmento.

Por isso, para cada cliente do golden set mostramos duas coisas:
1. **A recomendação do bandit** — sorteada 200 vezes a partir da posterior atual (já que o Thompson Sampling é estocástico), reportando a % de vezes que cada canal seria escolhido. Como é uma leitura da posterior atual (sem chamar `partial_fit`), rodar esta célula não altera o estado do bandit salvo.
2. **A taxa de conversão esperada por canal *para aquele segmento específico***, usando a mesma estimativa contextual (`contextual_rate`) da Etapa 3 — isso não influencia a decisão do bandit hoje, mas permite avaliar se a decisão do bandit *faria sentido* para aquele cliente em particular, e identificar onde uma versão contextual (próximo passo natural) traria mais ganho.

In [6]:
golden_set = pd.DataFrame([
    {'cliente': 'Cliente 1', 'idade': 25, 'age_segment': 'jovem',  'poutcome': 'unknown', 'previous': 0,
     'descricao': 'Jovem, nunca contatado antes'},
    {'cliente': 'Cliente 2', 'idade': 42, 'age_segment': 'adulto', 'poutcome': 'failure', 'previous': 2,
     'descricao': 'Adulto, campanha anterior falhou (2 contatos prévios)'},
    {'cliente': 'Cliente 3', 'idade': 63, 'age_segment': 'senior', 'poutcome': 'success', 'previous': 1,
     'descricao': 'Sênior, campanha anterior teve sucesso'},
    {'cliente': 'Cliente 4', 'idade': 38, 'age_segment': 'adulto', 'poutcome': 'failure', 'previous': 5,
     'descricao': 'Adulto, muitos contatos anteriores, todos sem sucesso'},
    {'cliente': 'Cliente 5', 'idade': 29, 'age_segment': 'jovem',  'poutcome': 'other', 'previous': 1,
     'descricao': 'Jovem, resultado anterior não classificado (other)'},
])

N_DRAWS = 200

def draw_recommendation_distribution(n_draws=N_DRAWS):
    """Sorteia n_draws vezes da posterior ATUAL do bandit (sem treinar/atualizar nada)
    e devolve a % de vezes que cada braço seria recomendado."""
    draws = [bandit.predict() for _ in range(n_draws)]
    counts = pd.Series(draws).value_counts(normalize=True)
    return {arm: round(float(counts.get(arm, 0.0)) * 100, 1) for arm in arms}


rows = []
for _, cliente in golden_set.iterrows():
    recomendacao_pct = draw_recommendation_distribution()
    canal_mais_recomendado = max(recomendacao_pct, key=recomendacao_pct.get)
    taxa_cellular = contextual_rate(cliente, 'cellular')
    taxa_telephone = contextual_rate(cliente, 'telephone')
    melhor_canal_para_segmento = 'cellular' if taxa_cellular >= taxa_telephone else 'telephone'

    rows.append({
        'cliente': cliente['cliente'],
        'descricao': cliente['descricao'],
        'oferta_recomendada': canal_mais_recomendado,
        '%_cellular': recomendacao_pct['cellular'],
        '%_telephone': recomendacao_pct['telephone'],
        'conv_esperada_cellular_segmento': round(taxa_cellular, 4),
        'conv_esperada_telephone_segmento': round(taxa_telephone, 4),
        'melhor_canal_para_este_segmento': melhor_canal_para_segmento,
        'decisao_fez_sentido': canal_mais_recomendado == melhor_canal_para_segmento,
    })

golden_set_results = pd.DataFrame(rows)
golden_set_results.to_csv(GOLDEN_SET_PATH, index=False)
golden_set_results

,cliente,descricao,oferta_recomendada,%_cellular,%_telephone,conv_esperada_cellular_segmento,conv_esperada_telephone_segmento,melhor_canal_para_este_segmento,decisao_fez_sentido
0,Cliente 1,"Jovem, nunca contatado antes",cellular,99.0,1.0,0.1781,0.0916,cellular,True
1,Cliente 2,"Adulto, campanha anterior falhou (2 contatos p...",cellular,99.5,0.5,0.1051,0.1021,cellular,True
2,Cliente 3,"Sênior, campanha anterior teve sucesso",cellular,99.5,0.5,0.5368,0.2964,cellular,True
3,Cliente 4,"Adulto, muitos contatos anteriores, todos sem ...",cellular,100.0,0.0,0.1666,0.0998,cellular,True
4,Cliente 5,"Jovem, resultado anterior não classificado (ot...",cellular,97.5,2.5,0.1894,0.1641,cellular,True


### Leitura caso a caso

Nos 5 casos do golden set, `cellular` acaba sendo a melhor escolha também no nível do segmento específico (`melhor_canal_para_este_segmento` = `cellular` para todos, e `decisao_fez_sentido` = `True` para todos) — a recomendação do bandit global concorda com a decisão que um modelo contextual tomaria caso a caso, então não há contradição a apontar nesta amostra específica.

Isso é consistente com o que já vimos na Etapa 3: `cellular` domina a maior parte dos segmentos desta base, o que é o motivo de o bandit global convergir tão fortemente para ele (99%+ das recomendações). O caso com a margem mais estreita entre os dois canais é o **Cliente 2** (adulto, campanha anterior com falha, 2 contatos prévios): conversão esperada de 10,51% em `cellular` contra 10,21% em `telephone` — uma diferença de apenas 0,3 p.p., a menor dos cinco perfis. É esse tipo de perfil que mais se beneficiaria de uma versão contextual do bandit (`LinTS`/`LinUCB`): quando os dois canais estão tão próximos, vale a pena continuar testando ambos por mais tempo para esse segmento em vez de já convergir para `cellular` como o bandit global faz.

## Registrando a avaliação no MLflow

Registramos as métricas de avaliação e o golden set numa run **vinculada** à run do Notebook 3 (via `parent_run_id`), na mesma run de mesma experiment `datathon-bandit-canal` — assim dá para ver, na mesma tela do MLflow, o modelo treinado e sua avaliação lado a lado.

In [7]:
run_id = log_notebook_execution(
    notebook_name='04_Avaliacao_e_Golden_Set',
    etapa='etapa4_avaliacao',
    params={
        'test_size': 0.30,
        'n_golden_cases': len(golden_set) if 'golden_set' in locals() else 0,
        'validation_method': 'manual_review',
    },
    metrics={
        'conversao_baseline': evaluation_metrics.get('conversao_baseline', 0.0),
        'conversao_thompson': evaluation_metrics.get('conversao_thompson', 0.0),
        'lift_absoluto_pp': evaluation_metrics.get('lift_absoluto_pp', 0.0),
        'lift_relativo_pct': evaluation_metrics.get('lift_relativo_pct', 0.0),
    },
    artifacts={
        'evaluation_metrics': str(PROCESSED_DIR / 'evaluation_metrics.json'),
        'golden_set_results': str(GOLDEN_SET_PATH),
    },
    tags={'tipo': 'avaliacao', 'etapa': '4'}
)

print(f'\n✅ Avaliação logada: {run_id}')

🏃 View run 04_Avaliacao_e_Golden_Set_etapa4_avaliacao at: http://localhost:5002/#/experiments/2/runs/e01fe1ad0dd048439a95ed4ce3b3a21c
🧪 View experiment at: http://localhost:5002/#/experiments/2

✅ Avaliação logada: e01fe1ad0dd048439a95ed4ce3b3a21c


In [8]:
# ============================================================================
# ETAPA FINAL - ASSOCIAR AVALIAÇÃO AO MODELO
# ============================================================================

print('\n' + '='*70)
print('Associando Avaliação ao Modelo Registrado')
print('='*70 + '\n')

from mlflow.tracking import MlflowClient
from pathlib import Path

try:
    client = MlflowClient(tracking_uri="http://localhost:5002")
    
    # Ler o run_id do Notebook 03
    print("🔗 Lendo referência do Notebook 03...\n")
    
    run_id_file = Path('../data/processed/bank-term-deposit-subscription_eda/mlflow_run_id.txt')
    if run_id_file.exists():
        notebook_03_run_id = run_id_file.read_text(encoding='utf-8').strip()
        print(f"✅ Run do Notebook 03: {notebook_03_run_id}")
    else:
        print("❌ Arquivo mlflow_run_id.txt não encontrado")
        notebook_03_run_id = None
    
    # Adicionar tag ao modelo apontando para a run de avaliação
    if notebook_03_run_id and run_id:
        print(f"\n🏷️  Adicionando tag de rastreabilidade...\n")
        
        model_name = "thompson_sampling_bandit"
        
        try:
            # Adicionar tag no modelo com a run de avaliação
            client.set_model_version_tag(
                name=model_name,
                version="1",
                key="evaluation_run_id",
                value=run_id
            )
            print(f"✅ Tag adicionada ao modelo:")
            print(f"   evaluation_run_id: {run_id}")
        except Exception as e:
            print(f"⚠️  Erro ao adicionar tag: {e}")
        
        # Resumo de rastreabilidade
        print(f"\n" + "="*70)
        print("✅ RASTREABILIDADE COMPLETA")
        print("="*70)
        print(f"\n📊 Modelo: thompson_sampling_bandit v1")
        print(f"   └─ Treino (Notebook 03): {notebook_03_run_id}")
        print(f"   └─ Avaliação (Notebook 04): {run_id}")
        print(f"\n   Acesso: http://localhost:5002/#/models/thompson_sampling_bandit")
    else:
        print("⚠️  Não foi possível associar - faltam dados")

except Exception as e:
    print(f"❌ Erro ao associar: {e}")
    import traceback
    traceback.print_exc()


Associando Avaliação ao Modelo Registrado

🔗 Lendo referência do Notebook 03...

✅ Run do Notebook 03: 11c70ec0264747a9bcd51bb4757fdea5

🏷️  Adicionando tag de rastreabilidade...

✅ Tag adicionada ao modelo:
   evaluation_run_id: e01fe1ad0dd048439a95ed4ce3b3a21c

✅ RASTREABILIDADE COMPLETA

📊 Modelo: thompson_sampling_bandit v1
   └─ Treino (Notebook 03): 11c70ec0264747a9bcd51bb4757fdea5
   └─ Avaliação (Notebook 04): e01fe1ad0dd048439a95ed4ce3b3a21c

   Acesso: http://localhost:5002/#/models/thompson_sampling_bandit
